# Enabling Science

:::{figure} ../thumbnails/gdex_logo.png
:alt: NSF NCAR GDEX logo
:width: 300px
:::

---

## Introduction

Scientific progress in the atmospheric and geosciences increasingly depends on the ability to work with massive, heterogeneous datasets that span decades and the covers the globe. Using these datasets at scale is itself a research infrastructure challenge.

GDEX attempts to address this challenge by curating, providing rich metadata, utilizing formats that fit modern workflows. This notebook describes how GDEX's design choices translate directly into scientific capability.

## Analysis-Ready and AI-Ready Data

A dataset is *analysis-ready* when a researcher can go from data discovery to running code without first spending hours (or days) on reformatting, subsetting, and cleaning. It is *AI-ready* when it additionally meets the throughput, structure, and fitness documentation requirements of machine learning training pipelines. GDEX invests in both.

### What Makes Data Analysis-Ready?

| Property | Why It Matters |
|---|---|
| **Cloud-optimized format** (Zarr, kerchunk) | Supports random access and parallel reads without full downloads |
| **Consistent metadata** (CF Conventions) | Variables are named and unitized the same way across datasets |
| **Chunked storage** | Enables efficient access along any axis — time, level, space |
| **Consolidated metadata** | In the case of zarr and kerchunk, one metadata read instead of one per file across thousands of files |
| **Stable, versioned URLs** | Scripts written today still work next year |


:::{tip}
Use the GDEX API endpoint `GET /has_arco/{dsid}/` to check whether a dataset has a cloud-optimized Zarr store before choosing an access pathway.
:::

### Cloud-Optimized Formats: Zarr and Kerchunk

GDEX produces and hosts **Zarr** stores for select high-value datasets. Zarr stores data as a directory of compressed chunks rather than a single monolithic file, making it ideal for parallel and partial reads from object storage or HTTPS.

```python
import xarray as xr

# Stream ERA5 surface winds directly — no download
ds = xr.open_zarr(
    "https://data.gdex.ucar.edu/d633000/e5.oper.an.sfc.zarr/e5.oper.an.sfc.100u.zarr",
    consolidated=True,
)
```

For datasets that are not natively in Zarr, GDEX also produces **Kerchunk** (also see [VirtualiZarr](https://virtualizarr.readthedocs.io/)) reference files. These JSON files map the chunks inside existing NetCDF/HDF5 files so that the same lazy, parallel access pattern works — no data conversion required.

```python
import xarray
import kerchunk
filename = 'https://data.gdex.ucar.edu/d633000/kerchunk/meanflux/Mean_evaporation_rate.json'
ds = xarray.open_dataset(filename, engine='kerchunk')
ds['MER'] # Mean evaporation rate
```


:::{note}
[VirtualiZarr](https://virtualizarr.readthedocs.io/) is the actively maintained successor to Kerchunk and provides the same virtual chunk mapping with a cleaner API. Prefer VirtualiZarr for new workflows.
:::

### Chunking

Chunk shape determines which access patterns are fast. Typically, GDEX will use a pattern that can fit multiple use cases, but below are examples where one could optimize.

- **Time-series analysis**: large time chunks, small spatial chunks
- **Spatial snapshots / mapping**: small time chunks, large spatial chunks
- **ML training (global fields)**: full spatial slices per time step, contiguous along the sample axis

When working with a GDEX Zarr/kerchunk store, you can inspect chunk shape and rechunk in memory for your specific workflow:

```python
print(ds.chunks)  # inspect stored chunk shape

# For example, Rechunk for a time-series workflow if memory constrained
ds_rechunked = ds.chunk({"time": 100, "latitude": 10, "longitude": 10})
```

### AI-Ready: Feeding Machine Learning Pipelines

Large geoscience ML projects — weather forecasting models, downscaling networks, climate emulators — have strict data pipeline requirements: high throughput, deterministic shuffling, and labeled samples. GDEX datasets in Zarr format integrate cleanly with these workflows.

**PyTorch example** — wrapping a Zarr-backed xarray dataset as a `Dataset`:

```python
import numpy as np
import torch
from torch.utils.data import Dataset
import xarray as xr

class GDEXDataset(Dataset):
    def __init__(self, zarr_url, variable, time_slice=None):
        ds = xr.open_zarr(zarr_url, consolidated=True)
        da = ds[variable]
        if time_slice:
            da = da.sel(time=time_slice)
        self.data = da.values  # loads into memory; use .chunk() + dask for lazy

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32)
```

For datasets too large to fit in memory, pair Zarr with [Dask](https://dask.org/) to build lazy pipelines that process one chunk at a time during training.


:::{caution}
`self.data = da.values` loads the entire array into memory at initialization. For datasets larger than available RAM, keep the data lazy with `da.chunk({...})` and load individual samples inside `__getitem__`.
:::

### GDEX AI-Ready Collections

GDEX is actively expanding its holdings of analysis-ready and AI-ready datasets, with a focus on:

- **ERA5 reanalysis** — global atmosphere at hourly resolution, cloud-optimized Zarr available for select surface and pressure-level variables
- **CESM Large Ensemble** — multi-member climate model output for ML-based uncertainty quantification
- **Radar S-Pol subset** — gridded retrievals pre-regridded to common grids to simplify data fusion [https://gdex.ucar.edu/datasets/d694517/](https://gdex.ucar.edu/datasets/d694517/)

Check the [GDEX portal](https://gdex.ucar.edu) for the current list of datasets with cloud-optimized access.

:::{seealso}
The **Workflows** section of this cookbook demonstrates end-to-end science examples using GDEX datasets, including ERA5 reanalysis and climate model output.
:::

## Reproducible Research

By providing stable, versioned, citable datasets, GDEX makes it possible to reproduce published analyses years after the fact. When a paper cites a GDEX dataset DOI, any researcher in the world can access the exact same data used in that study.

## Lowering Barriers to Large-Scale Analysis

Some of the most scientifically important datasets are also the largest. ERA5 global reanalysis alone spans petabytes. GDEX removes the logistical barriers to working with these datasets by:

- Hosting data on infrastructure optimized for high-throughput access
- Providing cloud-optimized formats (Zarr, Kerchunk) that enable streaming analysis without full downloads
- Supporting compute-proximate workflows on NSF NCAR HPC systems

## Enabling Multi-Dataset Science

Because GDEX curates data to common standards, it becomes easier to combine datasets — for example, comparing reanalysis products or merging model output with observations. Consistent metadata and file formats reduce the data wrangling burden and let researchers focus on science.

## Supporting Data-Driven Discovery

The scale and diversity of GDEX collections make them well-suited for emerging methods in geoscience:

- **Machine learning** applications that require large training datasets
- **Long-term trend analysis** using multi-decadal observational records
- **Model evaluation** comparing simulations against observational benchmarks
- **Climate impact studies** combining atmospheric, oceanic, and land surface data

## Example: Streaming Data Directly into Python

Cloud-optimized formats hosted by GDEX allow researchers to open petabyte-scale datasets with a single line of Python — no full download required:

```python
import xarray as xr

# Open a cloud-optimized GDEX dataset
ds = xr.open_zarr("https://data.gdex.ucar.edu/d633000/e5.oper.an.sfc.zarr/e5.oper.an.sfc.100u.zarr", consolidated=True)
ds["VAR_100U"].sel(time="2020-01-01").plot()
```

This workflow is possible because GDEX invests in cloud-optimized formats and stable, high-bandwidth access infrastructure.


:::{seealso}
{doc}`services` — an overview of every GDEX access pathway, from THREDDS and OPeNDAP to the REST API and Globus.
:::

---